# JaxQSOFit Tutorial: SDSS Example + Broad-Line Measurements

This notebook demonstrates the main features of `jaxqsofit` using a real SDSS spectrum near:

```python
coord = SkyCoord(184.0307, -2.2383, unit='deg')
```

Recommended fitting mode: `config.inference.method = 'optax+nuts'`.
This runs the staged SVI/Optax MAP initializer, then performs full posterior sampling with NUTS.

It covers:
- Fetching spectrum with `astroquery`
- Running fits (`nuts`, `optax`, `optax+nuts`)
- Overriding priors with `PriorConfig`
- Measuring broad-line FWHM and luminosity from fitted components

Set `config.inference.method = "optax"` for staged SVI/Optax MAP optimization (continuum warm start, then full model). Set `config.inference.method = "optax+nuts"` to use that staged MAP point to initialize NUTS.


In [1]:
import jaxqsofit
import jaxqsofit.defaults as defaults
import numpy as np

/Users/colinburke/miniforge3/envs/jaxcpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from astroquery.sdss import SDSS
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.cosmology import FlatLambdaCDM

from jaxqsofit import (
    ContinuumConfig,
    HostConfig,
    InferenceConfig,
    JAXQSOFit,
    LineConfig,
    Observation,
    OutputConfig,
    PreprocessingConfig,
    PriorConfig,
    FitConfig,
    SpectroscopyData,
    build_default_prior_config,
)


## 1. Download one SDSS spectrum


In [3]:
coord = SkyCoord(184.0307, -2.2383, unit='deg')
xid = SDSS.query_region(coord, spectro=True, radius=5 * u.arcsec)
sp = SDSS.get_spectra(matches=xid[:1])[0]

tb = sp[1].data
lam = np.asarray(10 ** tb['loglam'], dtype=float)
flux = np.asarray(tb['flux'], dtype=float)
ivar = np.asarray(tb['ivar'], dtype=float)

err = np.full_like(flux, 1e-6)
m = np.isfinite(ivar) & (ivar > 0)
err[m] = 1.0 / np.sqrt(ivar[m])

z = float(sp[2].data['z'][0])
ra = float(coord.ra.deg)
dec = float(coord.dec.deg)

print(f'Nspec pixels: {lam.size}')
print(f'z = {z:.5f}')

plate = int(sp[0].header.get('plateid', 0))
mjd = int(sp[0].header.get('mjd', 0))
fiber = int(sp[0].header.get('fiberid', 0))
sdss_filename = f"{plate:04d}-{mjd}-{fiber:04d}"


Nspec pixels: 3810
z = 0.10061


## 2. Build a default prior config (auto-scaled from flux)

You can use defaults directly, or modify selected entries.


In [5]:
flux_rest = flux * (1 + z)
prior_config = build_default_prior_config(
        flux_rest,
        include_high_ionization_lines=False,
        include_elg_narrow_lines=False,
)

# Example override: slightly tighter PL slope and Fe normalization priors
prior_config.overrides['PL_slope'] = {'loc': -1.5, 'scale': 0.3, 'low': -3.5, 'high': 0.5}
prior_config.overrides['log_Fe_uv_norm'] = {'loc': np.log(max(1e-3 * np.median(np.abs(flux)), 1e-10)), 'scale': 0.04}
prior_config.overrides['log_Fe_op_over_uv'] = {'loc': 0.0, 'scale': 0.4}

# Robust line scale multipliers (optional)
prior_config.lines.dmu_scale_mult = 0.25
prior_config.lines.sig_scale_mult = 0.25
prior_config.lines.amp_scale_mult = 0.20

prior_config.overrides["log_host_aperture_scale"] = {
    "dist": "Normal",
    "loc": 0.0,
    "scale": 0.5,
}

prior_config.host.sfh_model = "flexible" # "delayed"


## 3. Run a fit

Recommended:
- `config.inference.method = 'optax+nuts'` (staged SVI/Optax initialization, then full NUTS posterior)

Other options:
- `config.inference.method = 'nuts'` (full posterior, slower initialization)
- `config.inference.method = 'optax'` (staged MAP only, fastest, no posterior samples)


In [ ]:
cfg = FitConfig(
    observation=Observation(
        object_id=sdss_filename,
        redshift=z,
        ra=ra,
        dec=dec,
        apply_mw_deredden=True,
    ),
    spectroscopy=SpectroscopyData(wave_obs=lam, fluxes=flux, errors=err),
    continuum=ContinuumConfig(
        fit_power_law=True,
        fit_feii=True,
        fit_balmer_continuum=True,
        fit_polynomial_tilt=True,
    ),
    host=HostConfig(enabled=True, dsps_ssp_fn='../tempdata.h5'),
    lines=LineConfig(enabled=True),
    inference=InferenceConfig(
        method='optax+nuts',
        map_steps=2000,
        learning_rate=1e-2,
        num_warmup=50,
        num_samples=100,
        num_chains=1,
    ),
    output=OutputConfig(output_path='.', save_result=False, plot_fig=True, save_fig=False),
    prior_config=prior_config,
)

q = JAXQSOFit(cfg)
result = q.fit()


/Users/colinburke/research/jaxqsofit/src/jaxqsofit/core.py:2581: RuntimeWarning: Mean of empty slice.
  tmp_SN = np.array([flux[ind5100].mean() / flux[ind5100].std(), flux[ind3000].mean() / flux[ind3000].std(), flux[ind1350].mean() / flux[ind1350].std()])
/Users/colinburke/miniforge3/envs/jaxcpu/lib/python3.12/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/colinburke/miniforge3/envs/jaxcpu/lib/python3.12/site-packages/numpy/_core/_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/colinburke/miniforge3/envs/jaxcpu/lib/python3.12/site-packages/numpy/_core/_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/colinburke/miniforge3/envs/jaxcpu/lib/python3.12/site-packages/numpy/_core/_methods.py:214: RuntimeWarning: invalid value encoun

In [ ]:
for k in ["PL_norm", "cont_norm", "log_frac_host", "frac_host",
          "log_stellar_mass", "log_host_aperture_scale", "host_aperture_scale"]:
    if hasattr(q, "init_stage1_samples") and k in q.init_stage1_samples:
        
        print(k, np.asarray(q.init_stage1_samples[k]).ravel())

In [ ]:
print(q.numpyro_samples.keys())
print(q.config.host.sfh_model)
print(q._fit_decompose_host)

In [ ]:
import numpy as np

logm = np.asarray(q.numpyro_samples["log_stellar_mass"])
print(np.nanmedian(logm), np.nanpercentile(logm, [16, 84]))

Estimate reduced Chi squared:

In [ ]:
resid = np.asarray(q.flux) - np.asarray(q.model_total)
sigma = np.asarray(q.err)

# include fitted jitter (recommended)
if q.numpyro_samples is not None:
    s = q.numpyro_samples
    frac_j = float(np.median(np.asarray(s.get("frac_jitter", 0.0))))
    add_j  = float(np.median(np.asarray(s.get("add_jitter", 0.0))))
    sigma = np.sqrt(sigma**2 + (frac_j*np.abs(np.asarray(q.model_total)))**2 + add_j**2)

m = np.isfinite(resid) & np.isfinite(sigma) & (sigma > 0)
z = resid[m] / sigma[m]

chi2 = float(np.sum(z**2))
chi2_per_pixel = float(np.mean(z**2))      # more stable than reduced chi2 here
wrms = float(np.sqrt(np.mean(z**2)))

print("chi2:", chi2)
print("chi2_per_pixel:", chi2_per_pixel)
print("wrms (normalized residual std):", wrms)


## 4. MCMC diagnostics (trace + corner)


In [ ]:
q.plot_mcmc_diagnostics(
    #param_names='all',
    do_trace=True,
    do_corner=True,
    max_vector_elems=2,
    corner_bins=25,
    corner_max_points=1500,
)


## 5. Measure broad-line FWHM and luminosity with posterior errors

`JAXQSOFit` provides:
- `line_profile_from_components(line_key)` for posterior-median component profiles
- `line_profile_from_draw(draw_index, line_key)` for one posterior draw
- `line_props(profile, wave=None)` -> `(fwhm_kms, integrated_area)`

The integrated area is in `10^-17 erg s^-1 cm^-2` units, so convert to luminosity with cosmology.


In [ ]:
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

def flux_to_luminosity(area_1e17, z):
    d_l_cm = cosmo.luminosity_distance(z).to(u.cm).value
    return area_1e17 * 1e-17 * 4.0 * np.pi * d_l_cm**2

amp_draws = np.asarray(q.pred_out['line_amp_per_component'])

for line_key in ["CIV_br", "MgII_br", "Hb_br", "Ha_br"]:
    fwhm_samp, logL_samp = [], []
    for i in range(amp_draws.shape[0]):
        prof = q.line_profile_from_draw(i, line_key)
        fwhm, area = q.line_props(prof)
        fwhm_samp.append(fwhm)
        if np.isfinite(area) and area > 0:
            logL_samp.append(np.log10(flux_to_luminosity(area, q.z)))
        else:
            logL_samp.append(np.nan)

    fwhm_samp = np.asarray(fwhm_samp, dtype=float)
    logL_samp = np.asarray(logL_samp, dtype=float)

    f16, f50, f84 = np.nanpercentile(fwhm_samp, [16, 50, 84])
    l16, l50, l84 = np.nanpercentile(logL_samp, [16, 50, 84])

    print(
        f"{line_key:8s}  "
        f"FWHM={f50:.1f} (+{f84-f50:.1f}/-{f50-f16:.1f}) km/s   "
        f"logL={l50:.3f} (+{l84-l50:.3f}/-{l50-l16:.3f})"
    )


...but don't forget to subtract the instrumental resolution in quadrature before reporting the FWHM values.

## 6. Inspect posterior samples (recommended with `optax+nuts`)

If you used `config.inference.method = 'nuts'` or `'optax+nuts'`, posterior samples are stored in:
- `q.numpyro_samples`


In [ ]:
if hasattr(q, 'numpyro_samples') and q.numpyro_samples is not None:
    keys = sorted(q.numpyro_samples.keys())
    print('Num posterior params:', len(keys))
    print('First 20 keys:', keys)
else:
    print('No NumPyro samples available (likely config.inference.method=optax).')

## 7. Quick component diagnostics


In [ ]:
print('max data        :', np.nanmax(q.flux))
print('max total model :', np.nanmax(q.model_total))
print('max PL          :', np.nanmax(q.f_pl_model))
print('max host        :', np.nanmax(q.host))
print('max FeII        :', np.nanmax(q.f_fe_mgii_model + q.f_fe_balmer_model))
print('max Balmer cont :', np.nanmax(q.f_bc_model))
print('max lines       :', np.nanmax(q.f_line_model))
